# FineBio 3D Reconstruction Pipeline

This notebook implements a 3D reconstruction approach for FineBio videos:
1. **Multi-view geometry**: Use FPV + TPV camera calibration to reconstruct 3D scene
2. **3D object detection**: Triangulate YOLO detections from both views → 3D positions
3. **3D features**: Extract spatial relationships (distances, angles, interactions) in 3D space
4. **Action segmentation**: Use 3D features for better verb/manipulated/affected object detection

## Why 3D?
- **Better spatial understanding**: True 3D positions vs 2D bboxes
- **Occlusion handling**: Multiple views → complete 3D scene
- **Robust manipulation detection**: 3D hand-object distances more reliable
- **Spatial relationships**: Distance/angle features in 3D space

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import zipfile
import json
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass
from collections import Counter
import ultralytics

# Set paths
DATASET_PATH = Path("/home/nyan/FineBio")
CAMERA_POSES_PATH = DATASET_PATH / "14 misc" / "finebio_camera_poses.zip"
ANNOTATIONS_DIR = DATASET_PATH / "01 annotations"
ACTION_DIR = ANNOTATIONS_DIR / "finebio_action_annotations"

# Load YOLO model (fine-tuned on FineBio)
YOLO_MODEL_PATH = "/home/nyan/k-project/runs/detect/runs/finebio_yolo26/yolo26n_joint3/weights/best.pt"
yolo_model = ultralytics.YOLO(YOLO_MODEL_PATH)

print("Loaded YOLO model:", YOLO_MODEL_PATH)

Loaded YOLO model: /home/nyan/k-project/runs/detect/runs/finebio_yolo26/yolo26n_joint3/weights/best.pt


In [2]:
# Load camera calibration data (from aruco_detection.ipynb)

def load_camera_poses_data(poses_zip_path):
    """Load camera poses data from zip file"""
    import tempfile
    intrinsic_params = {}
    marker_points = {}
    extrinsics = {}
    fp_poses = {}
    temp_dir = Path(tempfile.mkdtemp())
    
    try:
        with zipfile.ZipFile(poses_zip_path, 'r') as z:
            z.extractall(temp_dir)
            
            # Load intrinsic parameters
            intrinsic_dir = temp_dir / "finebio_camera_poses" / "intrinsic_parameters"
            if intrinsic_dir.exists():
                for npz_file in intrinsic_dir.glob("*.npz"):
                    data = np.load(npz_file, allow_pickle=True)
                    intrinsic_params[npz_file.stem] = {k: data[k] for k in data.keys()}
            
            # Load marker points and extrinsics
            marker_dir = temp_dir / "finebio_camera_poses" / "third_person_camera_poses"
            if marker_dir.exists():
                for date_dir in marker_dir.iterdir():
                    if date_dir.is_dir():
                        marker_file = date_dir / "params" / "marker_points.npy"
                        if marker_file.exists():
                            marker_points[date_dir.name] = np.load(marker_file, allow_pickle=True)
                        
                        extrinsics_dir = date_dir / "extrinsics"
                        if extrinsics_dir.exists():
                            extrinsics[date_dir.name] = {}
                            for extr_file in extrinsics_dir.glob("*.npz"):
                                data = np.load(extr_file, allow_pickle=True)
                                extrinsics[date_dir.name][extr_file.stem] = {k: data[k] for k in data.keys()}
            
            # Load first-person camera poses
            fp_poses_dir = temp_dir / "finebio_camera_poses" / "first_person_camera_poses"
            if fp_poses_dir.exists():
                for npz_file in fp_poses_dir.glob("*.npz"):
                    data = np.load(npz_file, allow_pickle=True)
                    fp_poses[npz_file.stem] = {k: data[k] for k in data.keys()}
            
        return intrinsic_params, marker_points, extrinsics, fp_poses, temp_dir
    except Exception as e:
        print(f"Error loading camera poses: {e}")
        return None, None, None, None, None

# Load calibration data
print("Loading camera calibration data...")
intrinsic_params, marker_points, extrinsics, fp_poses, temp_dir = load_camera_poses_data(CAMERA_POSES_PATH)

if intrinsic_params:
    print(f"Loaded {len(intrinsic_params)} camera intrinsic parameter sets")
if extrinsics:
    print(f"Loaded extrinsics for {len(extrinsics)} dates")
if fp_poses:
    print(f"Loaded {len(fp_poses)} first-person camera poses")

Loading camera calibration data...
Loaded 2 camera intrinsic parameter sets
Loaded extrinsics for 10 dates
Loaded 226 first-person camera poses


In [3]:
# Helper: Get camera matrices for an instance
# (We'll need to match instance_id to date/calibration data)

def get_camera_matrices(instance_id: str, intrinsic_params, extrinsics, fp_poses):
    """
    Get camera intrinsic and extrinsic matrices for FPV and TPV cameras.
    
    Returns:
        K_fpv, K_tpv: Intrinsic matrices (3x3)
        R_fpv, t_fpv: FPV rotation (3x3) and translation (3x1)
        R_tpv, t_tpv: TPV rotation (3x3) and translation (3x1)
    """
    # TODO: Match instance_id to date/calibration set
    # For now, use first available calibration
    
    # Get intrinsics (assume same for all cameras of same type)
    if 'fpv' in intrinsic_params:
        K_fpv_data = intrinsic_params['fpv']
        K_fpv = K_fpv_data.get('camera_matrix', np.eye(3))
    else:
        # Fallback: use default camera matrix (will need proper calibration)
        K_fpv = np.array([[800, 0, 320], [0, 800, 240], [0, 0, 1]], dtype=np.float32)
    
    if 'tpv' in intrinsic_params or len(intrinsic_params) > 0:
        # Use first available intrinsic
        first_key = list(intrinsic_params.keys())[0]
        K_tpv_data = intrinsic_params[first_key]
        K_tpv = K_tpv_data.get('camera_matrix', np.eye(3))
    else:
        K_tpv = np.array([[800, 0, 320], [0, 800, 240], [0, 0, 1]], dtype=np.float32)
    
    # Get extrinsics (rotation + translation)
    # FPV: from fp_poses (first-person camera)
    if fp_poses and len(fp_poses) > 0:
        first_fpv = list(fp_poses.values())[0]
        R_fpv = first_fpv.get('R', np.eye(3))
        t_fpv = first_fpv.get('t', np.zeros((3, 1)))
    else:
        R_fpv = np.eye(3)
        t_fpv = np.zeros((3, 1))
    
    # TPV: from extrinsics (third-person camera)
    if extrinsics and len(extrinsics) > 0:
        first_date = list(extrinsics.keys())[0]
        first_cam = list(extrinsics[first_date].keys())[0]
        tpv_data = extrinsics[first_date][first_cam]
        R_tpv = tpv_data.get('R', np.eye(3))
        t_tpv = tpv_data.get('t', np.zeros((3, 1)))
    else:
        R_tpv = np.eye(3)
        t_tpv = np.array([[0], [0], [2.0]])  # TPV typically 2m away
    
    return K_fpv, K_tpv, R_fpv, t_fpv, R_tpv, t_tpv

print("Camera matrix helper function defined")

Camera matrix helper function defined


In [ ]:
# 3D Triangulation: Match YOLO detections across views → 3D positions

def triangulate_point(p1, p2, K1, R1, t1, K2, R2, t2):
    """
    Triangulate 3D point from 2D correspondences in two views.
    
    Args:
        p1, p2: 2D points (x, y) in image coordinates
        K1, K2: Intrinsic matrices (3x3)
        R1, t1: Rotation (3x3) and translation (3x1) for camera 1
        R2, t2: Rotation (3x3) and translation (3x1) for camera 2
    
    Returns:
        P: 3D point (X, Y, Z) in world coordinates
    """
    # Projection matrices: P = K [R|t]
    P1 = K1 @ np.hstack([R1, t1])
    P2 = K2 @ np.hstack([R2, t2])
    
    # Triangulate using DLT (Direct Linear Transform)
    A = np.array([
        p1[0] * P1[2, :] - P1[0, :],
        p1[1] * P1[2, :] - P1[1, :],
        p2[0] * P2[2, :] - P2[0, :],
        p2[1] * P2[2, :] - P2[1, :]
    ])
    
    # Solve Ax = 0 using SVD
    _, _, Vt = np.linalg.svd(A)
    P_homogeneous = Vt[-1, :]
    
    # Convert from homogeneous to 3D
    if abs(P_homogeneous[3]) > 1e-6:
        P = P_homogeneous[:3] / P_homogeneous[3]
    else:
        P = P_homogeneous[:3]
    
    return P


def match_detections_across_views(dets_fpv, dets_tpv, K_fpv, R_fpv, t_fpv, K_tpv, R_tpv, t_tpv, 
                                   iou_threshold=0.3, max_reprojection_error=200.0):
    """
    Match YOLO detections from FPV and TPV views based on:
    1. Class name (must match)
    2. Geometric consistency (reprojection error)
    3. IoU in epipolar geometry
    
    Returns:
        matches: List of (det_fpv_idx, det_tpv_idx, P_3d) tuples
    """
    matches = []
    
    for i, det_fpv in enumerate(dets_fpv):
        if det_fpv is None:
            continue
        class_fpv = det_fpv.get('class_name', '')
        bbox_fpv = det_fpv.get('bbox', [])  # [x1, y1, x2, y2]
        center_fpv = [(bbox_fpv[0] + bbox_fpv[2]) / 2, (bbox_fpv[1] + bbox_fpv[3]) / 2]
        
        best_match = None
        best_error = float('inf')
        
        for j, det_tpv in enumerate(dets_tpv):
            if det_tpv is None:
                continue
            class_tpv = det_tpv.get('class_name', '')
            
            # Must be same class
            if class_fpv != class_tpv:
                continue
            
            bbox_tpv = det_tpv.get('bbox', [])
            center_tpv = [(bbox_tpv[0] + bbox_tpv[2]) / 2, (bbox_tpv[1] + bbox_tpv[3]) / 2]
            
            # Triangulate 3D point
            try:
                P_3d = triangulate_point(
                    np.array(center_fpv), np.array(center_tpv),
                    K_fpv, R_fpv, t_fpv, K_tpv, R_tpv, t_tpv
                )
                
                # Check reprojection error
                # Project 3D point back to both views
                P_hom = np.append(P_3d, 1.0)
                proj_fpv = (K_fpv @ (R_fpv @ P_3d + t_fpv.ravel()))[:2]
                proj_tpv = (K_tpv @ (R_tpv @ P_3d + t_tpv.ravel()))[:2]
                
                error_fpv = np.linalg.norm(proj_fpv - center_fpv)
                error_tpv = np.linalg.norm(proj_tpv - center_tpv)
                total_error = error_fpv + error_tpv
                
                if total_error < max_reprojection_error and total_error < best_error:
                    best_match = (i, j, P_3d)
                    best_error = total_error
            except:
                continue
        
        if best_match:
            matches.append(best_match)
    
    return matches

print("Triangulation and matching functions defined")

Triangulation and matching functions defined


In [5]:
# Build 3D features from reconstructed scene

@dataclass
class Object3D:
    """3D object representation"""
    class_name: str
    position: np.ndarray  # (3,) XYZ in world coordinates
    confidence: float
    bbox_fpv: Optional[List[float]] = None
    bbox_tpv: Optional[List[float]] = None

def compute_3d_features(objects_3d: List[Object3D], hands_3d: List[Object3D]) -> np.ndarray:
    """
    Extract 3D spatial features from reconstructed scene.
    
    Features:
    1. Object positions (normalized)
    2. Hand-object distances (for each hand-object pair)
    3. Object-object distances (for nearby objects)
    4. Hand-object angles (direction vectors)
    5. Object counts per class (in 3D space)
    
    Returns:
        features: (D,) feature vector
    """
    if not objects_3d:
        return np.zeros(100)  # Placeholder dimension
    
    # 1. Object positions (normalize by scene center)
    positions = np.array([obj.position for obj in objects_3d])
    scene_center = positions.mean(axis=0) if len(positions) > 0 else np.zeros(3)
    positions_normalized = positions - scene_center
    
    # 2. Hand-object distances
    hand_object_distances = []
    for hand in hands_3d:
        for obj in objects_3d:
            dist = np.linalg.norm(hand.position - obj.position)
            hand_object_distances.append(dist)
    
    # 3. Object-object distances (for nearby pairs)
    object_distances = []
    for i, obj1 in enumerate(objects_3d):
        for obj2 in objects_3d[i+1:]:
            dist = np.linalg.norm(obj1.position - obj2.position)
            if dist < 0.5:  # Only nearby objects
                object_distances.append(dist)
    
    # 4. Object class counts (in 3D)
    class_counts = Counter([obj.class_name for obj in objects_3d])
    
    # 5. Hand-object direction vectors (normalized)
    hand_directions = []
    for hand in hands_3d:
        for obj in objects_3d[:5]:  # Top 5 closest
            direction = (obj.position - hand.position)
            norm = np.linalg.norm(direction)
            if norm > 1e-6:
                direction_normalized = direction / norm
                hand_directions.extend(direction_normalized)
    
    # Combine features (simplified - you can expand this)
    features = np.concatenate([
        positions_normalized.flatten()[:30],  # Max 10 objects * 3 coords
        np.array(hand_object_distances)[:20],
        np.array(object_distances)[:20],
        np.array(hand_directions)[:15],
        np.array(list(class_counts.values()))[:15]
    ])
    
    # Pad or truncate to fixed dimension
    target_dim = 100
    if len(features) < target_dim:
        features = np.pad(features, (0, target_dim - len(features)))
    else:
        features = features[:target_dim]
    
    return features

print("3D feature extraction function defined")

3D feature extraction function defined


In [6]:
# Main pipeline: Reconstruct 3D scene from FPV + TPV frames

def reconstruct_3d_frame(frame_fpv, frame_tpv, yolo_model, K_fpv, R_fpv, t_fpv, K_tpv, R_tpv, t_tpv, conf=0.25):
    """
    Reconstruct 3D scene from synchronized FPV and TPV frames.
    
    Steps:
    1. Run YOLO on both frames
    2. Match detections across views
    3. Triangulate 3D positions
    4. Extract 3D features
    
    Returns:
        objects_3d: List of Object3D
        features_3d: Feature vector
    """
    # 1. Run YOLO on both views
    results_fpv = yolo_model(frame_fpv, conf=conf, verbose=False)
    results_tpv = yolo_model(frame_tpv, conf=conf, verbose=False)
    
    # Parse detections
    dets_fpv = []
    for r in results_fpv:
        boxes = r.boxes
        for i in range(len(boxes)):
            cls_id = int(boxes.cls[i])
            class_name = yolo_model.names[cls_id]
            conf_score = float(boxes.conf[i])
            bbox = boxes.xyxy[i].cpu().numpy()  # [x1, y1, x2, y2]
            dets_fpv.append({
                'class_name': class_name,
                'bbox': bbox.tolist(),
                'confidence': conf_score
            })
    
    dets_tpv = []
    for r in results_tpv:
        boxes = r.boxes
        for i in range(len(boxes)):
            cls_id = int(boxes.cls[i])
            class_name = yolo_model.names[cls_id]
            conf_score = float(boxes.conf[i])
            bbox = boxes.xyxy[i].cpu().numpy()
            dets_tpv.append({
                'class_name': class_name,
                'bbox': bbox.tolist(),
                'confidence': conf_score
            })
    
    # 2. Match detections and triangulate
    matches = match_detections_across_views(
        dets_fpv, dets_tpv, K_fpv, R_fpv, t_fpv, K_tpv, R_tpv, t_tpv
    )
    
    # 3. Build 3D objects
    objects_3d = []
    hands_3d = []
    
    for i_fpv, i_tpv, P_3d in matches:
        det_fpv = dets_fpv[i_fpv]
        det_tpv = dets_tpv[i_tpv]
        
        obj_3d = Object3D(
            class_name=det_fpv['class_name'],
            position=P_3d,
            confidence=(det_fpv['confidence'] + det_tpv['confidence']) / 2,
            bbox_fpv=det_fpv['bbox'],
            bbox_tpv=det_tpv['bbox']
        )
        
        if 'hand' in det_fpv['class_name'].lower():
            hands_3d.append(obj_3d)
        else:
            objects_3d.append(obj_3d)
    
    # 4. Extract 3D features
    features_3d = compute_3d_features(objects_3d, hands_3d)
    
    return objects_3d, hands_3d, features_3d

print("3D reconstruction pipeline function defined")

3D reconstruction pipeline function defined


In [ ]:
# Test on a single instance with debugging

# Get video paths (reuse from yolo26.ipynb)
def resolve_video_paths(instance_id: str, view: str = "tpv") -> List[Path]:
    """Return all matching video paths for a given instance id and view."""
    paths: List[Path] = []
    
    if view == "tpv":
        tpv_dirs = [
            DATASET_PATH / "05 finebio_videos_tpv_train" / "finebio_videos",
            DATASET_PATH / "06 finebio_videos_tpv_valid" / "finebio_videos",
            DATASET_PATH / "08 finebio_videos_tpv_test" / "finebio_videos",
        ]
        trials = ["T1", "T2", "T3", "T4", "T5"]
        for d in tpv_dirs:
            for t in trials:
                cand = d / f"{instance_id}_{t}.mp4"
                if cand.exists():
                    paths.append(cand)
    elif view == "fpv":
        fpv_dirs = [
            DATASET_PATH / "09 finebio_videos_fpv_train" / "finebio_videos",
            DATASET_PATH / "11 finebio_videos_fpv_valid" / "finebio_videos",
            DATASET_PATH / "04 finebio_videos_fpv_test",
        ]
        trials = ["T1", "T2", "T3", "T4", "T5"]
        for d in fpv_dirs:
            cand_single = d / f"{instance_id}.mp4"
            if cand_single.exists():
                paths.append(cand_single)
            for t in trials:
                cand = d / f"{instance_id}_{t}.mp4"
                if cand.exists():
                    paths.append(cand)
    
    return paths

# Test on P03_02_01
instance_id = "P03_02_01"
fpv_paths = resolve_video_paths(instance_id, view="fpv")
tpv_paths = resolve_video_paths(instance_id, view="tpv")

if fpv_paths and tpv_paths:
    print(f"Found FPV: {fpv_paths[0]}")
    print(f"Found TPV: {tpv_paths[0]}")
    
    # Get camera matrices
    K_fpv, K_tpv, R_fpv, t_fpv, R_tpv, t_tpv = get_camera_matrices(
        instance_id, intrinsic_params, extrinsics, fp_poses
    )
    
    print(f"\nCamera matrices:")
    print(f"K_fpv shape: {K_fpv.shape}, K_tpv shape: {K_tpv.shape}")
    print(f"R_fpv shape: {R_fpv.shape}, t_fpv shape: {t_fpv.shape}")
    print(f"R_tpv shape: {R_tpv.shape}, t_tpv shape: {t_tpv.shape}")
    
    # Load frames
    cap_fpv = cv2.VideoCapture(str(fpv_paths[0]))
    cap_tpv = cv2.VideoCapture(str(tpv_paths[0]))
    
    # Sample frame at 5 seconds
    cap_fpv.set(cv2.CAP_PROP_POS_MSEC, 5000)
    cap_tpv.set(cv2.CAP_PROP_POS_MSEC, 5000)
    ret_fpv, frame_fpv = cap_fpv.read()
    ret_tpv, frame_tpv = cap_tpv.read()
    
    if ret_fpv and ret_tpv:
        print(f"\nFrame shapes: FPV {frame_fpv.shape}, TPV {frame_tpv.shape}")
        
        # Debug: Check YOLO detections first
        print("\nRunning YOLO on both views...")
        results_fpv = yolo_model(frame_fpv, conf=0.25, verbose=False)
        results_tpv = yolo_model(frame_tpv, conf=0.25, verbose=False)
        
        dets_fpv = []
        for r in results_fpv:
            boxes = r.boxes
            for i in range(len(boxes)):
                cls_id = int(boxes.cls[i])
                class_name = yolo_model.names[cls_id]
                conf_score = float(boxes.conf[i])
                bbox = boxes.xyxy[i].cpu().numpy()
                dets_fpv.append({
                    'class_name': class_name,
                    'bbox': bbox.tolist(),
                    'confidence': conf_score
                })
        
        dets_tpv = []
        for r in results_tpv:
            boxes = r.boxes
            for i in range(len(boxes)):
                cls_id = int(boxes.cls[i])
                class_name = yolo_model.names[cls_id]
                conf_score = float(boxes.conf[i])
                bbox = boxes.xyxy[i].cpu().numpy()
                dets_tpv.append({
                    'class_name': class_name,
                    'bbox': bbox.tolist(),
                    'confidence': conf_score
                })
        
        print(f"FPV detections: {len(dets_fpv)}")
        if dets_fpv:
            fpv_classes = Counter([d['class_name'] for d in dets_fpv])
            print(f"  Classes: {dict(fpv_classes)}")
        
        print(f"TPV detections: {len(dets_tpv)}")
        if dets_tpv:
            tpv_classes = Counter([d['class_name'] for d in dets_tpv])
            print(f"  Classes: {dict(tpv_classes)}")
        
        # Try matching with relaxed thresholds
        print("\nAttempting to match detections...")
        matches = match_detections_across_views(
            dets_fpv, dets_tpv, K_fpv, R_fpv, t_fpv, K_tpv, R_tpv, t_tpv,
            max_reprojection_error=200.0  # More lenient
        )
        print(f"Found {len(matches)} matches")
        
        if len(matches) > 0:
            print("\nSample matches:")
            for i, (i_fpv, i_tpv, P_3d) in enumerate(matches[:3]):
                print(f"  Match {i+1}: {dets_fpv[i_fpv]['class_name']} at 3D {P_3d}")
        
        # Reconstruct 3D scene
        print("\nReconstructing 3D scene...")
        objects_3d, hands_3d, features_3d = reconstruct_3d_frame(
            frame_fpv, frame_tpv, yolo_model, K_fpv, R_fpv, t_fpv, K_tpv, R_tpv, t_tpv,
            conf=0.25
        )
        
        print(f"\nReconstructed {len(objects_3d)} objects and {len(hands_3d)} hands in 3D")
        print(f"3D feature dimension: {len(features_3d)}")
        if objects_3d:
            print(f"Sample object: {objects_3d[0].class_name} at {objects_3d[0].position}")
        if hands_3d:
            print(f"Sample hand: {hands_3d[0].class_name} at {hands_3d[0].position}")
    else:
        print("Failed to read frames")
    
    cap_fpv.release()
    cap_tpv.release()
else:
    print(f"Could not find videos for {instance_id}")

Found FPV: /home/nyan/FineBio/04 finebio_videos_fpv_test/P03_02_01.mp4
Found TPV: /home/nyan/FineBio/08 finebio_videos_tpv_test/finebio_videos/P03_02_01_T1.mp4
Reconstructing 3D scene from sample frames...
Reconstructed 0 objects and 0 hands in 3D
3D feature dimension: 100
